# Wolt menu data

Inspects and cleans the Wolt menus collected with `wolt_menus.py` (Copenhagen bounds, 2 km grid, collected 2026-10-22).

The items are split into separate dataframes, so each group can be inspected on its own:

| dataframe | what |
|---|---|
| `zero_price` | items priced 0 kr (cutlery, info rows, free extras) |
| `non_food` | bags, cutlery and other packaging that costs money |
| `bundles` | menus, combos, deals and "dish + drink" offers |
| `drinks` | soft drinks, beer, wine, coffee, tea, juice, shakes, ... |
| `food` | what is left: the food-specific menu items |

The order matters: an item only lands in the first group it matches, e.g. "Pasta + sodavand 0,33 l" is a bundle, not a drink.
Drinks and bundles are found with keyword rules on the category and item name, so they are heuristics: the check cells under each step show what was caught and what slipped through.

## Imports

In [18]:
import re

import pandas as pd
from config import WOLT_DATA_DIR

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)

## Load

In [20]:
AREA = "vesterbro"

venues = pd.read_csv(WOLT_DATA_DIR / AREA / "venues.csv")
items_raw = pd.read_csv(WOLT_DATA_DIR / AREA / "items.csv")
print(f"{len(venues)} venues, {len(items_raw)} items from {items_raw['wolt_id'].nunique()} venues")

191 venues, 9967 items from 191 venues


In [21]:
venues.head()

,wolt_id,name,slug,address,lat,lon
0,6a71e83b2c4b0028d09cf99e,1001 Nat Pizza,1001-nat-pizza,"Istedgade 75, 1650 København",55.668622,12.551987
1,62b2e9f227ad244b3050085c,Aamanns Deli & Takeaway Carlsberg Byen,aamanns-deli-takeaway-carlsbergbyen,"Aamanns Genbo, Kildepladsen 8",55.665628,12.536944
2,5ecd26c84881b0f97887f575,Al-Diwan,al-diwan,Vesterbrogade 94,55.672340,12.548070
3,5fd79172976cd377a071fb28,Abu Kebab,and-kebab,Kalvebod Brygge 59,55.661932,12.561373
4,65aa5c1f8f058b4a89efbe4a,Anh's Spiseri og Takeaway,anhs-spiseri-og-takeaway,Vesterbrogade 134,55.670915,12.542265


In [22]:
items_raw.head()

,wolt_id,venue,category,item,description,price_dkk
0,6a71e83b2c4b0028d09cf99e,1001 Nat Pizza,Burger,Klassisk okseburger,"Sesambolle med fyld af oksebøf, dressing, ketchup, icebe...",88.0
1,6a71e83b2c4b0028d09cf99e,1001 Nat Pizza,Hovedretter,En halv grillkylling,NaN,108.0
2,6a71e83b2c4b0028d09cf99e,1001 Nat Pizza,Burger,Maxi Cheeseburger,"Briochebolle med fyld af ca. 300 grams oksebøf, ost, sal...",128.0
3,6a71e83b2c4b0028d09cf99e,1001 Nat Pizza,Pizzaer,Gorgonzola,"Tomatsauce, ost, gorgonzola, champignon, løg og rucola.",115.0
4,6a71e83b2c4b0028d09cf99e,1001 Nat Pizza,Pizzaer,Caprese,"Tomatsauce, ost, kødstrimler, paprika og bearnaisesauce.",115.0


## Helpers

All matching is done on lowercased, stripped text (`category_text_normalised`, `item_text_normalised`).

Matching uses python's `re` instead of `Series.str.contains`: pandas 3 runs `str.contains` on pyarrow's regex engine, where `\b` only knows ASCII letters, so `\bøl\b` never matches "øl".

In [ ]:
for d in items_raw.description:
    print(d)

Sesambolle med fyld af oksebøf, dressing, ketchup, icebergsalat, løgringe, tomat og agurk.
nan
Briochebolle med fyld af ca. 300 grams oksebøf, ost, salat, tomat, løg og agurk.
Tomatsauce, ost, gorgonzola, champignon, løg og rucola.
Tomatsauce, ost, kødstrimler, paprika og bearnaisesauce.
Tomatsauce, ost, kylling, pepperoni og paprika.
Tomatsauce, ost, spinat, oliven, gorgonzola, champignon og hvidløg.
Serveret med pommes frites, salat og remoulade.
Tomatsauce, ost, kebab, kylling, pepperoni eller kalkunskinke, salat og dressing.
Oksekødsret serveret med pommes frites, hummus, salat og brød.
nan
Tyrkisk brød med fyld af valgfri protein, salat, tomat, agurk og creme fraiche.
Box med fyld af kebab, salat, pommes frites og dressing.
Tomatsauce, ost og kalkunskinke.
Serveret med pommes frites, salat, dressing og remoulade.
nan
Tomatsauce, ost, kebab og champignon.
Græsk bøf med stegte løg, pommes frites, jalapeños, salat og bearnaisesauce.
Tomatsauce, ost, rejer, tun, paprika og løg.
Tomats

In [30]:
def normalise(s: pd.Series) -> pd.Series:
    return s.fillna("").str.lower().str.strip()


def matches(s: pd.Series, pattern: str) -> pd.Series:
    rx = re.compile(pattern)
    return s.map(lambda x: bool(rx.search(x)))


def litres(text: pd.Series) -> pd.Series:
    """The volume in the item name in litres ('0,33 l', '33 cl', '330 ml'), NaN if there is none."""
    volume = text.str.extract(r"(\d+(?:[.,]\d+)?)\s?(l|cl|ml)\b")
    return volume[0].str.replace(",", ".").astype(float) * volume[1].map({"l": 1, "cl": 0.01, "ml": 0.001})


items = items_raw.assign(category_text_normalised=normalise(items_raw["category"]), item_text_normalised=normalise(items_raw["item"]))

In [31]:
items.category_text_normalised.value_counts()

category_text_normalised
sodavand            600
tilbehør            418
hovedretter         332
pizzaer             324
forretter           286
                   ... 
bland-selv-salat      1
alkoholfri vine       1
combo side            1
enchiladas combo      1
enchiladas            1
Name: count, Length: 591, dtype: int64

In [32]:
items.head()

,wolt_id,venue,category,item,description,price_dkk,category_text_normalised,item_text_normalised
0,6a71e83b2c4b0028d09cf99e,1001 Nat Pizza,Burger,Klassisk okseburger,"Sesambolle med fyld af oksebøf, dressing, ketchup, icebe...",88.0,burger,klassisk okseburger
1,6a71e83b2c4b0028d09cf99e,1001 Nat Pizza,Hovedretter,En halv grillkylling,NaN,108.0,hovedretter,en halv grillkylling
2,6a71e83b2c4b0028d09cf99e,1001 Nat Pizza,Burger,Maxi Cheeseburger,"Briochebolle med fyld af ca. 300 grams oksebøf, ost, sal...",128.0,burger,maxi cheeseburger
3,6a71e83b2c4b0028d09cf99e,1001 Nat Pizza,Pizzaer,Gorgonzola,"Tomatsauce, ost, gorgonzola, champignon, løg og rucola.",115.0,pizzaer,gorgonzola
4,6a71e83b2c4b0028d09cf99e,1001 Nat Pizza,Pizzaer,Caprese,"Tomatsauce, ost, kødstrimler, paprika og bearnaisesauce.",115.0,pizzaer,caprese


## 1. Items priced 0 kr

In [42]:
is_zero = items["price_dkk"] == 0
zero_price = items[is_zero]
items = items[~is_zero]
print(f"{len(zero_price)} items priced 0 kr removed, {len(items)} left")
zero_price["category_text_normalised"].value_counts().head(15)

125 items priced 0 kr removed, 9954 left


category_text_normalised
engangsservice                  104
sådan fungerer wolt catering      6
cutlery                           6
tacos                             2
                                  2
small dishes                      1
fries & dips                      1
build-your-own                    1
engangsbestik og servietter       1
wedos egne dressinger             1
Name: count, dtype: int64

## 2. Non-food (bags, cutlery, packaging)

In [43]:
NON_FOOD = r"engangs|bestik|cutlery|\bposer?\b|bærepose|\bpant\b|emballage|^service$"

is_non_food = matches(items["category_text_normalised"], NON_FOOD)
non_food = items[is_non_food]
items = items[~is_non_food]
print(f"{len(non_food)} non-food items, {len(items)} left")
non_food["category_text_normalised"].value_counts().head(10)

77 non-food items, 9877 left


category_text_normalised
engangsservice                                  75
pose                                             1
engangsservice(huske at bestille spisepinde)     1
Name: count, dtype: int64

## 3. Bundles

Menus, combos, deals and offers, found by category name ("Menuer", "Combos", "Tilbud", "Børnemenu", ...) or item name ("Big Mac Menu", "Ramen + Soft Drink").

In [44]:
BUNDLE_CATEGORY = r"menu|combo|bundle|tilbud|\bdeals?\b|happy meal"
BUNDLE_ITEM = BUNDLE_CATEGORY + r"|\+.*(cola|pepsi|fanta|sprite|sodavand|soft ?drink|drik|øl\b)"

by_category = matches(items["category_text_normalised"], BUNDLE_CATEGORY)
by_item = matches(items["item_text_normalised"], BUNDLE_ITEM)
is_bundle = by_category | by_item
bundles = items[is_bundle]
items = items[~is_bundle]
print(f"{len(bundles)} bundles ({by_category.sum()} by category, {(by_item & ~by_category).sum()} only by item name), {len(items)} left")
bundles["category_text_normalised"].value_counts().head(20)

934 bundles (744 by category, 190 only by item name), 8943 left


category_text_normalised
menuer                    260
combos                     82
sushimenuer                36
max classics               32
menus                      28
bundles                    26
børnemenuer                22
green family               22
frokostmenu                20
kylling                    20
mini menu                  18
børnemenu                  15
frokost                    13
menu                       13
wedo combo deals           13
tilbudsmenuer              12
chicken tenders            12
oksekød                    12
happy meal®                12
kylling, fisk & veggie     12
Name: count, dtype: int64

Check: bundles found only by the item name

In [45]:
bundles[~matches(bundles["category_text_normalised"], BUNDLE_CATEGORY)].sample(20, random_state=0)[["venue", "category", "item", "price_dkk"]]

,venue,category,item,price_dkk
6178,MAX Premium Burgers Dybbølsbro,MAX CLASSICS,Cheese 'n' Bacon menu,143.0
6056,MAX Premium Burgers Hovedbanegården,GREEN FAMILY,Crispy Supreme Sandwich menu,139.0
6395,McDonald's Hovedbanegården,Oksekød,Big Tasty® Bacon Menu,128.0
6119,MAX Premium Burgers Hovedbanegården,CHICKEN TENDERS,Chicken Tenders Hot Habanero 3 stk menu,137.0
6198,MAX Premium Burgers Dybbølsbro,KYLLING,Grilled Chicken Buffalo menu,125.0
6047,MAX Premium Burgers Hovedbanegården,MAX CLASSICS,Frisco menu,127.0
2381,El Amigo Vesterbro,Vegetarisk/Vegansk,Burrito menu,150.0
6199,MAX Premium Burgers Dybbølsbro,GREEN FAMILY,Crispy Supreme Buffalo menu,139.0
6292,MAX Premium Burgers Dybbølsbro,KYLLING,Chicken Nuggets 6 stk menu,109.0
960,Börger,Burgere,BÖF Bearnasie Burger Menu,190.0


## 4. Drinks

An item is a drink if
- its category is a drinks category (sodavand, øl, vin, kaffe, juice, smoothies, ...), or
- its name is a drink (latte, espresso, tea, milkshake, ...) or has a drink-sized volume (0.2 - 2 l, e.g. "0,33 l"), unless the name or category says it is food (ice cream sold per litre, dips in ml, curries in "650 ml", ...)

In [46]:
DRINK_CATEGORY = (
    r"sodavand|\bvand\b|\bøl\b|\bbeers?\b|drik|\bdrinks?\b|softdrink|juice|smoothie|shake|lemonade|limonade|\bsaft\b|"
    r"(rød|hvid|rosé|mouserende |hede|\b)vine?\b|\brosé?\b|bobler|champagne|cocktail|spiritus|alkohol|cider|\bsake\b|"
    r"kaffe|coffee|espresso|\blatte|matcha|\bte\b|\btea\b|iste\b|chai|bubble|kakao|kombucha|\blassi\b|ayran|ramune|vitamin well|beverage|\bsoda"
)
DRINK_ITEM = r"\blatte\b|cappuccino|espresso|\bamericano\b|macchiato|\bmocha\b|frap|\bkaffe\b|\bcoffee\b|\btea\b|\bte\b|smoothie|milkshake|lemonade|\bjuice\b"
NOT_DRINK = (
    r"\bis\b|isboks|softice|flødeis|ice ?cream|gelato|ben & jerry|\bdip|dressing|sauce|mayo|bowl|suppe|soup|kage|cake|cookie|"
    r"burger|pizza|sandwich|med ris|raita|curry|vindalo|paneer|masala|oats|\bball\b|\bbar\b"
)

by_category = matches(items["category_text_normalised"], DRINK_CATEGORY)
looks_like_food = matches(items["item_text_normalised"], NOT_DRINK) | matches(items["category_text_normalised"], NOT_DRINK)
by_item = (matches(items["item_text_normalised"], DRINK_ITEM) | litres(items["item_text_normalised"]).between(0.2, 2.0)) & ~looks_like_food
is_drink = by_category | by_item
drinks = items[is_drink]
items = items[~is_drink]
print(f"{len(drinks)} drinks ({by_category.sum()} by category, {(by_item & ~by_category).sum()} only by item name), {len(items)} left")
drinks["category_text_normalised"].value_counts().head(25)

2378 drinks (2287 by category, 91 only by item name), 6565 left


category_text_normalised
sodavand          604
øl                210
vand              152
drinks             94
varme drikke       84
rødvin             71
kolde drikke       69
kaffe & te         69
hvidvin            64
drikkevarer        55
drikkevare         43
kaffe              37
soft drinks        33
juice              31
dåse sodavand      30
milkshakes         28
juicer             24
beer               22
frapino            21
milkshake          20
ready to drink     20
vin                19
lemonade           18
iskaffe            15
signatur-juice     15
Name: count, dtype: int64

Check: drinks found only by the item name

In [47]:
drinks[~matches(drinks["category_text_normalised"], DRINK_CATEGORY)].sample(20, random_state=0)[["venue", "category", "item", "price_dkk"]]

,venue,category,item,price_dkk
1271,Burger Palace,Kiosk,Jack Daniel's Whiskey 70 cl,349.0
2464,Espresso House Fisketorvet,Frapino,Coffee Caramel Frapino Std,74.0
8323,Sober Boba Bubble Tea,Cheese Foam 0.7L,Cheese Foam Rose Milk Tea,101.0
8136,Smagsløget Vesterbro,Red Bull,"Red Bull 0,25 l",30.0
8467,Starbucks® Fisketorvet,Cappuccino,Iced Cappuccino,52.0
2615,Espresso House Tivoli,Frapino,Coffee Caramel Frapino Std,74.0
8301,Sober Boba Bubble Tea,Brown Sugar,Brown Sugar Pearl Latte,82.0
8193,Smagsløget Vesterbro,Red Bull,"Red Bull Sugarfree 0,25 l.",30.0
8345,Sober Boba Bubble Tea,Cheese Foam 0.7L,Cheese Foam Assam Milk Tea,101.0
9589,Warung,Noda,"Noda Mango Passionsfrugt 0,33 l",39.0


## 5. Food

In [48]:
items.head()

,slug,venue,category,item,description,price_dkk,category_text_normalised,item_text_normalised
0,1001-nat-pizza,1001 Nat Pizza,Burger,Klassisk okseburger,"Sesambolle med fyld af oksebøf, dressing, ketchup, icebe...",88.0,burger,klassisk okseburger
1,1001-nat-pizza,1001 Nat Pizza,Hovedretter,En halv grillkylling,NaN,108.0,hovedretter,en halv grillkylling
2,1001-nat-pizza,1001 Nat Pizza,Burger,Maxi Cheeseburger,"Briochebolle med fyld af ca. 300 grams oksebøf, ost, sal...",128.0,burger,maxi cheeseburger
3,1001-nat-pizza,1001 Nat Pizza,Pizzaer,Gorgonzola,"Tomatsauce, ost, gorgonzola, champignon, løg og rucola.",115.0,pizzaer,gorgonzola
4,1001-nat-pizza,1001 Nat Pizza,Pizzaer,Caprese,"Tomatsauce, ost, kødstrimler, paprika og bearnaisesauce.",115.0,pizzaer,caprese


In [49]:
food = items.drop(columns=["category_text_normalised", "item_text_normalised"])
print(f"{len(food)} food items from {food['slug'].nunique()} venues")
food.head()

6565 food items from 192 venues


,slug,venue,category,item,description,price_dkk
0,1001-nat-pizza,1001 Nat Pizza,Burger,Klassisk okseburger,"Sesambolle med fyld af oksebøf, dressing, ketchup, icebe...",88.0
1,1001-nat-pizza,1001 Nat Pizza,Hovedretter,En halv grillkylling,NaN,108.0
2,1001-nat-pizza,1001 Nat Pizza,Burger,Maxi Cheeseburger,"Briochebolle med fyld af ca. 300 grams oksebøf, ost, sal...",128.0
3,1001-nat-pizza,1001 Nat Pizza,Pizzaer,Gorgonzola,"Tomatsauce, ost, gorgonzola, champignon, løg og rucola.",115.0
4,1001-nat-pizza,1001 Nat Pizza,Pizzaer,Caprese,"Tomatsauce, ost, kødstrimler, paprika og bearnaisesauce.",115.0


Check: food items that still look like drinks (what the rules missed)

In [50]:
LEFTOVER = r"\bcola\b|coca|pepsi|fanta|\bøl\b|\bvin\b|juice|kaffe|latte|smoothie|\btea\b|\bte\b|0,33|\bcl\b"
leftovers = items[matches(items["item_text_normalised"], LEFTOVER)]
print(f"{len(leftovers)} food items match drink words ({len(leftovers) / len(food):.2%} of food)")
leftovers.sample(min(20, len(leftovers)), random_state=0)[["venue", "category", "item", "price_dkk"]]

4 food items match drink words (0.06% of food)


,venue,category,item,price_dkk
7741,Restaurant Aquavit,Smørrebrød,Platte,149.0
8974,The South Indian Vesterbro,Foretter,13. Kings Platter 65,149.0
6979,Namaste Fisketorvet,Forretter,Tandoori Platter,149.0
3518,Great China Istedgade,Dry-wok retter,一百零七 Tørstegt and med øl,188.0


In [51]:
food["category"].value_counts().head(30)

category
Tilbehør           411
Hovedretter        327
Pizzaer            324
Forretter          290
Dyppelse           207
Dips               139
Sides              130
Dim Sum            127
Pizze              120
Pizza              111
Nigiri             111
Sandwiches         102
Uramaki            102
Salater             93
Smørrebrød          88
Kaburimaki          79
Desserter           73
Burgere             61
Hosomaki            61
Futomaki            56
Pizza               52
Sticks              52
Pasta               50
TILLBEHØR & DIP     50
Burger              44
Snacks              44
Kylling             40
Kyllingeretter      38
Bagværk & Kage      36
Lammeretter         36
Name: count, dtype: int64

In [52]:
food.describe().round(2) # only describes the price, which is numbers

,price_dkk
count,6565.00
mean,101.13
std,159.32
min,1.00
25%,50.00
50%,95.00
75%,135.00
max,8000.00


In [54]:
food.description.value_counts().tail(30)

description
Bionda salat, rucolasalat, frisk mozzarella, avocado, semitørrede tomater, mandler, pinjekerner, persille, basilikum og vores hjemmelavet pesto. Serveres med friskbagt fuldkorns-sesam-brød             1
Marineret kylling, bionda salat, rucolasalat, frisk mozzarella, avocado, semitørrede tomater, mandler, pinjekerner, persille og vores hjemmelavet pesto. Serveres med friskbagt fuldkorns-sesam-brød.    1
Hjemmelavet Chili Mayo lavet på Anaheim chili, kogt op i 5 timer med akaciehonning. Chilien er mild og frisk.                                                                                            1
3 stk. Nuggets af kyllingebryst med 1 valgfri dip.                                                                                                                                                       1
6 stk. Nuggets af kyllingebryst med 1 valgfri dip.                                                                                                                              

## 6. Ingredients

Most descriptions are ingredient lists ("Tomatsauce, ost, kylling, pepperoni og paprika."), so the ingredients are found by splitting the description:
- keep the first sentence, plus later sentences about sides and toppings ("Serveres med brød.", "Toppet med sesam."); drop the rest (CO2e, allergens, kcal, marketing)
- drop a leading "Med" / "With"
- split on `,`, `og`, `and`, `samt`, `&`, ` - ` and an inner "med" / "toppet med" ("rød karry med kokosmælk")

Descriptions written as prose ("Lam i lækker karrysauce med friske grøntsager") are not broken up, and variants are not merged (tomat / tomater / cherrytomater, avocado / avokado, Danish / English): the check cells show what that leaves.

In [55]:
SENTENCE = r"[.!?\n]+"
SIDES = r"^(serveres|serveret|served|toppet|toppes|topped)( med| with)?\s+"
LEADING = r"^(med|with|lavet med|made with)\s+"
SEPARATOR = (
    r",|\s+og\s+|\s+and\s+|\s+samt\s+|\s*&\s*|\s+[-–]\s+|"
    r"\s+(?:toppet |toppes |topped |serveret |served )?(?:med|with)\s+"
)


def ingredients(description: str) -> list[str]:
    """The ingredients in a description: the first sentence plus any 'Serveres/Toppet med ...' sentences, split into parts."""
    sentences = [s.strip() for s in re.split(SENTENCE, description) if s.strip()]
    kept = sentences[:1] + [s for s in sentences[1:] if re.match(SIDES, s)]
    parts = []
    for sentence in kept:
        sentence = re.sub(LEADING, "", re.sub(SIDES, "", sentence))
        parts += [p.strip(" -–:;()'\"") for p in re.split(SEPARATOR, sentence)]
    return [p for p in parts if p]


food = food.assign(ingredients=normalise(food["description"]).map(ingredients))
n_ingredients = food["ingredients"].str.len()
print(f"{(n_ingredients >= 2).sum()} of {len(food)} food items have 2+ ingredients, {(n_ingredients == 1).sum()} have 1, {(n_ingredients == 0).sum()} have none (no description)")
food.sample(15, random_state=0)[["venue", "item", "description", "ingredients"]]

4547 of 6565 food items have 2+ ingredients, 709 have 1, 1309 have none (no description)


,venue,item,description,ingredients
4637,Joe & The Juice Axel Towers,JOE's Protein Bar,JOE's Protein Bar,[joe's protein bar]
9461,Wami Sushi Istedgade,Laks & avocado Hosomaki,NaN,[]
6095,MAX Premium Burgers Hovedbanegården,Raising MAX's Sauce,Raising MAX's Sauce dip. Tilgængelig i tre forskellige s...,[raising max's sauce dip]
4567,Jiao Hao Chi,Rice with black pepper beef,Rice with black pepper beef,"[rice, black pepper beef]"
1225,Burger Palace,Grillkylling uden tilbehør (450 g),Halv grillkylling uden tilbehør.,[halv grillkylling uden tilbehør]
9956,Ych!,Pizza 4,"Sesambund, tomat, mozzarella, skalotteløg, kødsauce, gro...","[sesambund, tomat, mozzarella, skalotteløg, kødsauce, gr..."
5023,Kanagawa Sushi Vesterbro,65. Rispapir Laks (8 stk),"Avocado, agurk, marineret rødløg, laks og blandet salat....","[avocado, agurk, marineret rødløg, laks, blandet salat, ..."
736,Bella Gastronomia Italiana,Penne al Salmone,"Med penne, røget laks, løg, broccoli og flødesauce. Serv...","[penne, røget laks, løg, broccoli, flødesauce, friskbagt..."
1953,DEJ Kødbyen,Højland (H),"Mozzarella, Grana Padano, gedeost, macinato af okse og l...","[mozzarella, grana padano, gedeost, macinato af okse, la..."
8021,Rossopomodoro Fisketorvet,"Nocellara Olives (V, VG)",green olives with garlic and chilli,"[green olives, garlic, chilli]"


Check: the most common ingredients

In [56]:
ingredient_counts = food["ingredients"].explode().value_counts()
print(f"{len(ingredient_counts)} unique ingredients")
ingredient_counts.head(40)

5280 unique ingredients


ingredients
tomat              684
ost                501
avocado            419
agurk              390
salat              389
løg                372
mozzarella         354
ris                332
tomatsauce         225
champignon         185
kylling            175
pesto              173
rucola             154
forårsløg          138
hvidløg            137
grøntsager         135
parmesan           134
dressing           128
koriander          117
chili              115
rødløg             115
sesam              115
laks               114
chilimayo          110
bacon              108
pepperoni           96
spinat              94
kartofler           93
cherrytomater       90
æg                  89
peberfrugt          86
pommes frites       83
tomater             81
cheddar             81
tun                 80
skinke              79
valgfri protein     79
paprika             75
oliven              73
purløg              73
Name: count, dtype: int64

Check: long ingredients (prose the split could not break up)

In [57]:
all_ingredients = food[["venue", "item", "ingredients"]].explode("ingredients").dropna()
is_long = all_ingredients["ingredients"].str.split().str.len() > 3
print(f"{is_long.sum()} of {len(all_ingredients)} ingredients are longer than 3 words")
all_ingredients[is_long].sample(20, random_state=0)

1665 of 22610 ingredients are longer than 3 words


,venue,item,ingredients
6565,Mexihagen Frederiksberg,Mexihagen bowl,dine yndlingstoppings fra det mexicanske køkken
1920,Deep Indisk Restaurant,30. Karahi Gosht,autentiske krydderier lavet i karahi
2204,Dodo's Noodles,Homemade Pan Fried Veggie Dumplings (V) 素饺子,sprøde pandestegte dumplings fyldt
3383,Græsk Smag,'NYT' Græsk klassisk risengrød (Rizogalo),er en græsk risengrød
9758,WokXpress - Vesterbro,Pad Thai,valgfrit protein i hjemmelavet pad thai sauce
5732,Madam Bum Thai Takeaway,Phat Priew Wan,stegte grøntsager i sursød sauce
4012,Hubb Kitchens Vesterbro,Korean,gochuang mayo serveret i en luftig briochebolle
2446,Espresso House Fisketorvet,Surdejsbolle Smør & Ost,surdejsbolle fra eget bageri
3555,Grillaz,Steak Showdown,vores fan favorit er tilbage
3600,Halifax Burgers Vesterbro,Ketchup,let krydret hjemmelavet ketchup
